# 07 · Evaluation, calibration and ablation reports

Compute results from real saved predictions. No claimed benchmark accuracy is bundled. The small examples below only verify metric definitions.

In [1]:
from pathlib import Path
import sys, json, os
ROOT = Path.cwd()
if not (ROOT / 'pyproject.toml').exists(): ROOT = ROOT.parent
assert (ROOT / 'pyproject.toml').exists(), 'Open this notebook from CineScope/notebooks or CineScope.'
sys.path.insert(0, str(ROOT / 'backend'))
sys.path.insert(0, str(ROOT / 'scripts'))
os.chdir(ROOT)
(ROOT / 'artifacts').mkdir(exist_ok=True)
SEED = 42

In [2]:
from cinescope.evaluation import ranking_metrics,expected_calibration_error
example=ranking_metrics({'u':[1,2,3]},{'u':{1,3}},k=3,catalog_size=10)
print('Metric-function example only:',example)
assert example['recall_at_k']==1
assert abs(example['precision_at_k']-2/3)<1e-9
print('Calibration example:',expected_calibration_error([0,1,1],[.1,.7,.9]))

Metric-function example only: {'eligible_users': 1, 'recall_at_k': 1.0, 'ndcg_at_k': 0.9197207891481876, 'precision_at_k': 0.6666666666666666, 'coverage': 0.3, 'note': 'Users with no heldout relevant items excluded; no-recommendation users with relevant items count as zero.'}
Calibration example: 0.16666666666666666


In [3]:
import pandas as pd
files=[ROOT/'artifacts/sentiment_tfidf_metrics.json',ROOT/'artifacts/transformer/test_metrics.json',ROOT/'artifacts/absa/test_metrics.json',ROOT/'artifacts/als_metrics.json']
for path in files:
    if path.exists():
        print(path.name);display(json.loads(path.read_text()))
    else:
        print(path.name, '→ not trained/evaluated yet')

sentiment_tfidf_metrics.json
{'selected_C': 2.0, 'validation': {'accuracy': 0.8906064209274673, 'macro_precision': 0.8888439518037283, 'macro_recall': 0.8892171789282757, 'macro_f1': 0.8890269338548586, 'confusion_matrix': [[1946, 271], [281, 2548]], 'n': 5046, 'roc_auc': 0.9591199977423085}, 'test': {'accuracy': 0.900520182265414, 'macro_precision': 0.9005206637778278, 'macro_recall': 0.900524696440962, 'macro_f1': 0.9005199726261817, 'confusion_matrix': [[11148, 1212], [1255, 11184]], 'n': 24799, 'roc_auc': 0.9631457109399371}, 'train_rows': 19733, 'seed': 42, 'sklearn_version': '1.7.2', 'screened_high_risk_train': 0, 'probability_note': 'Logistic probabilities; evaluate calibration before treating them as confidence.'}
test_metrics.json → not trained/evaluated yet
test_metrics.json → not trained/evaluated yet
als_metrics.json → not trained/evaluated yet


In [4]:
rank_path=ROOT/'artifacts/als_ranked';test_path=ROOT/'artifacts/als_test';train_path=ROOT/'artifacts/als_train'
if rank_path.exists() and test_path.exists() and train_path.exists():
    ranked=pd.read_parquet(rank_path).sort_values(['userId','rank'])
    test=pd.read_parquet(test_path);train=pd.read_parquet(train_path)
    relevant=test[test.rating>=4].groupby('userId').movieId.apply(set).to_dict()
    predictions=ranked.groupby('userId').movieId.apply(list).to_dict()
    catalog_size=pd.concat([train.movieId,test.movieId]).nunique()
    als_report=ranking_metrics(predictions,relevant,k=10,catalog_size=catalog_size)
    # Popularity uses TRAIN counts only and excludes each user's TRAIN items.
    popular=train.groupby('movieId').size().sort_values(ascending=False).index.tolist()
    seen=train.groupby('userId').movieId.apply(set).to_dict()
    baseline={u:[m for m in popular if m not in seen.get(u,set())][:10] for u in relevant}
    popularity_report=ranking_metrics(baseline,relevant,k=10,catalog_size=catalog_size)
    result={'ALS':als_report,'train-popularity':popularity_report}
    (ROOT/'artifacts/ranking_metrics.json').write_text(json.dumps(result,indent=2));display(pd.DataFrame(result).T)
else:
    print('Run notebook 05 to generate ranking predictions and temporal train/test splits.')

Run notebook 05 to generate ranking predictions and temporal train/test splits.


In [5]:
import time, numpy as np
from cinescope.reviews import ReviewAnalyzer
model=ReviewAnalyzer()
texts=[r['text'] for r in json.loads((ROOT/'data/demo/reviews.json').read_text())]
latencies=[]
for text in texts:
    model.score.cache_clear()
    start=time.perf_counter();model.analyze(text);latencies.append((time.perf_counter()-start)*1000)
print({'method':model.model_name,'synthetic_short_texts':len(texts),'latency_ms_p50':float(np.percentile(latencies,50)),'latency_ms_p95':float(np.percentile(latencies,95))})
print('This is a local microbenchmark, not a throughput/SLA claim.')

{'method': 'tfidf-logistic-regression', 'synthetic_short_texts': 36, 'latency_ms_p50': 1.7463974999714083, 'latency_ms_p95': 2.264268749996745}
This is a local microbenchmark, not a throughput/SLA claim.


## Required final report table
| Task | Metrics | Split and comparison |
|---|---|---|
| Binary sentiment | Macro-F1, accuracy, ROC-AUC, calibration | Official test; TF-IDF vs DistilBERT |
| Aspect sentiment | Per-aspect macro-F1, coverage, agreement | Human labels; movie-disjoint splits |
| Recommendations | Recall@10, NDCG@10, precision@10, coverage, diversity | Per-user temporal; popularity vs ALS vs content vs hybrid |
| Review screening | Precision/recall against audited labels; false positives | Author/source holdout; heuristic vs labeled model if available |
| Summaries | ROUGE against references + human evidence/factuality/polarity assessment | Movie-disjoint reviews; extractive vs neural |
| Conversation | Relevant movie recall, constraint compliance, groundedness | Human query set; lexical vs semantic retrieval |
| Speech | WER/CER, latency | Speaker/accent-diverse recorded queries with consent |
| Big-data pipeline | Runtime, shuffle, memory, partitions | Same input on 1/2/4 executors; report actual cluster |

Add ablations by removing ALS, themes, recency weighting, screening and diversity independently. Do not compare models tested on different splits or silently remove unscored users/items. Set the random seed and save package/model versions and data provenance with every experiment.